# Week 3: Clustering for Netflix Recommendations
## KMeans, Agglomerative, DBSCAN + KNN Collaborative Filtering

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
%matplotlib inline

## Load Netflix Ratings Data

In [ ]:
# Load user-movie ratings
url = "https://raw.githubusercontent.com/ucla-anderson-SSAI/SSAI/main/netflix_ratings.csv"
df = pd.read_csv(url)

# Identify user column and movie columns
user_col = df.columns[0]
movie_cols = list(df.columns[1:])

print(f"Users: {len(df)}")
print(f"Movies: {len(movie_cols)}")
df.head()

In [ ]:
# Prepare feature matrix (ratings)
X = df[movie_cols].values

# Handle missing values
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X)

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

print(f"Feature matrix: {X_scaled.shape}")

## Elbow Method for Optimal K

In [ ]:
# Test k values from 2 to 10
k_values = range(2, 11)
inertias = []
silhouette_scores = []

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))

# Plot elbow curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(k_values, inertias, marker='o', linewidth=2)
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Inertia')
ax1.set_title('Elbow Method: Inertia')
ax1.grid(True, alpha=0.3)

ax2.plot(k_values, silhouette_scores, marker='o', linewidth=2, color='green')
ax2.set_xlabel('Number of Clusters (k)')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Score by k')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Optimal k based on silhouette
optimal_k = k_values[np.argmax(silhouette_scores)]
print(f"\nOptimal k (by silhouette): {optimal_k}")

## Clustering Algorithm Comparison

In [ ]:
k = 5  # Choose k for comparison

# KMeans
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
labels_kmeans = kmeans.fit_predict(X_scaled)

# Agglomerative
agg = AgglomerativeClustering(n_clusters=k, linkage='ward')
labels_agg = agg.fit_predict(X_scaled)

# DBSCAN (different paradigm, may find different number of clusters)
dbscan = DBSCAN(eps=0.5, min_samples=5)
labels_dbscan = dbscan.fit_predict(X_scaled)

# Calculate metrics
sil_kmeans = silhouette_score(X_scaled, labels_kmeans)
sil_agg = silhouette_score(X_scaled, labels_agg)

# DBSCAN silhouette (exclude noise points)
if len(set(labels_dbscan)) > 1 and -1 in labels_dbscan:
    mask = labels_dbscan != -1
    if mask.sum() > 1:
        sil_dbscan = silhouette_score(X_scaled[mask], labels_dbscan[mask])
    else:
        sil_dbscan = None
else:
    sil_dbscan = silhouette_score(X_scaled, labels_dbscan)

results = pd.DataFrame({
    'Algorithm': ['KMeans', 'Agglomerative', 'DBSCAN'],
    'N_Clusters': [
        len(set(labels_kmeans)),
        len(set(labels_agg)),
        len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
    ],
    'Silhouette': [sil_kmeans, sil_agg, sil_dbscan]
})

print("\nClustering Algorithm Comparison:")
print(results)

## Cluster Profiles (Top Movies per Cluster)

In [ ]:
# Use KMeans clustering
labels = labels_kmeans

# Calculate average ratings per cluster
for cluster_id in range(k):
    cluster_mask = labels == cluster_id
    cluster_data = X_imputed[cluster_mask]
    
    # Get average rating for each movie
    avg_ratings = cluster_data.mean(axis=0)
    
    # Top 5 movies
    top_indices = np.argsort(avg_ratings)[::-1][:5]
    top_movies = [(movie_cols[i], avg_ratings[i]) for i in top_indices]
    
    print(f"\nCluster {cluster_id} (n={cluster_mask.sum()}):")
    for movie, rating in top_movies:
        print(f"  {movie}: {rating:.2f}")

## KNN Collaborative Filtering

In [ ]:
# Build KNN model for recommendations
n_neighbors = 10
knn = NearestNeighbors(n_neighbors=n_neighbors+1, metric='cosine')
knn.fit(X_scaled)

# Example: Recommend rating for user 0, movie 0
user_idx = 0
movie_idx = 0

# Find similar users
distances, indices = knn.kneighbors([X_scaled[user_idx]])
similar_indices = indices[0][1:]  # Exclude user themselves
similar_distances = distances[0][1:]

# Weighted average of similar users' ratings
weights = 1 - similar_distances
similar_ratings = X_imputed[similar_indices, movie_idx]
predicted_rating = np.average(similar_ratings, weights=weights)

print(f"User {user_idx}, Movie '{movie_cols[movie_idx]}'")
print(f"Actual rating: {X_imputed[user_idx, movie_idx]:.2f}")
print(f"Predicted rating: {predicted_rating:.2f}")
print(f"\nBased on {n_neighbors} similar users")

## Cluster Visualization (PCA)

In [ ]:
from sklearn.decomposition import PCA

# Reduce to 2D for visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# Plot clusters
plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels_kmeans, cmap='tab10', alpha=0.6, s=50)
plt.colorbar(scatter, label='Cluster')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.title('KMeans Clustering Visualization (PCA)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()